In [3]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from catboost import CatBoostClassifier

from Utils.DataUtils import build_ae_dataloaders

from Models.FraudModel import FraudMLP
from Models.AutoEncoder import AutoEncoder
from Models.EnsembleModel import (
    TorchBinaryProbWrapper,
    WeightedFraudEnsemble,
    StackingFraudMLP,
    StackedFraudEnsemble,
    CatBoostAEWrapper,
    collect_member_probs,
    collect_weighted_ensemble_probs,
    collect_stacked_ensemble_probs,
    collect_labels,
)

from Utils.TrainUtils import get_device, train_model

In [2]:
# We need to use the val_loader for training this time as if we keep training only on the train_loader 
# we might overfit bad. 
# But this means we will no longer be able to validate.

train_loader, val_loader, test_loader, input_dim = build_ae_dataloaders(batch_size=256)

[INFO] Project root: c:\Users\mengt\Documents\DeepLearning\DL_project
[INFO] Data dir: c:\Users\mengt\Documents\DeepLearning\DL_project\data
[INFO] Loading train data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)


In [4]:
# construct models
DEVICE = get_device()

# construct baseV6
model_base_v6 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    encoder=None,
    freeze_encoder=False,
).to(DEVICE)

model_base_v6_ckpt = torch.load("checkpoints/GatedMLP_V6/best.pt", map_location=DEVICE)
if "model_state_dict" in model_base_v6_ckpt:
    model_base_v6.load_state_dict(model_base_v6_ckpt["model_state_dict"])
else:
    model_base_v6.load_state_dict(model_base_v6_ckpt)

model_base_v6.eval()

# construct model ae16
autoencoder16  = AutoEncoder(
    input_dim=input_dim,
    latent_dim=16,
    hidden_dims=[128, 64],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt16 = torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE)
if "model_state_dict" in ae_ckpt16:
    autoencoder16.load_state_dict(ae_ckpt16["model_state_dict"])
else:
    autoencoder16.load_state_dict(ae_ckpt16)
autoencoder16.eval()

model_ae16 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=autoencoder16,
    freeze_encoder=True,
).to(DEVICE)

ckpt_model_ae16 = torch.load("checkpoints/GatedMLP_AE16_V4/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_model_ae16:
    model_ae16.load_state_dict(ckpt_model_ae16["model_state_dict"])
else:
    model_ae16.load_state_dict(ckpt_model_ae16)

model_ae16.eval()

# construct model ae256
autoencoder256 = AutoEncoder(
    input_dim=input_dim,
    latent_dim=256,
    hidden_dims=[128, 64, 32],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt256 = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)
ae_state_dict256 = ae_ckpt256["model_state_dict"] if "model_state_dict" in ae_ckpt256 else ae_ckpt256
autoencoder256.load_state_dict(ae_state_dict256)
autoencoder256.eval()

model_ae256 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    encoder=autoencoder256,
    freeze_encoder=True,
).to(DEVICE)

ckpt_model_ae256 = torch.load("checkpoints/GatedMLP_AE256_V1/best.pt", map_location=DEVICE)
state_dict_model_ae256 = ckpt_model_ae256["model_state_dict"] if "model_state_dict" in ckpt_model_ae256 else ckpt_model_ae256

model_ae256.load_state_dict(state_dict_model_ae256)
model_ae256.eval()

# construct catboost model
cat_model = CatBoostClassifier()
cat_model.load_model("checkpoints/best_catboost.cbm")

tree_member = CatBoostAEWrapper(
    cat_model=cat_model,
    autoencoder=autoencoder256
)

print("Base V6 ckpt config:", model_base_v6_ckpt.get("model_config"))
print("AE16 ckpt config   :", ckpt_model_ae16.get("model_config"))
print("AE256 ckpt config  :", ckpt_model_ae256.get("model_config"))

print("Loaded Base config :", model_base_v6.config)
print("Loaded AE16 config :", model_ae16.config)
print("Loaded AE256 config:", model_ae256.config)

Base V6 ckpt config: {'input_dim': 776, 'hidden_dims': [1024, 512, 256, 128, 64], 'gated': True, 'dropout': 0.1, 'use_norm': True, 'has_encoder': False, 'freeze_encoder': False, 'encoder_latent_dim': None}
AE16 ckpt config   : {'input_dim': 776, 'hidden_dims': [1024, 512, 256, 128, 64], 'gated': True, 'dropout': 0.1, 'use_norm': True, 'has_encoder': True, 'freeze_encoder': True, 'encoder_latent_dim': 16}
AE256 ckpt config  : {'input_dim': 776, 'hidden_dims': [1024, 512, 256, 128, 64], 'gated': True, 'dropout': 0.3, 'use_norm': True, 'has_encoder': True, 'freeze_encoder': True, 'encoder_latent_dim': 256}
Loaded Base config : {'input_dim': 776, 'hidden_dims': [1024, 512, 256, 128, 64], 'gated': True, 'dropout': 0.1, 'use_norm': True, 'has_encoder': False, 'freeze_encoder': False, 'encoder_latent_dim': None, 'use_recon_error_vector': False}
Loaded AE16 config : {'input_dim': 776, 'hidden_dims': [1024, 512, 256, 128, 64], 'gated': True, 'dropout': 0.1, 'use_norm': True, 'has_encoder': True

In [5]:
members = [
    TorchBinaryProbWrapper(model_base_v6),
    TorchBinaryProbWrapper(model_ae16),
    TorchBinaryProbWrapper(model_ae256),
    tree_member
]

batch = next(iter(val_loader))
x = batch[0].to(DEVICE)

with torch.no_grad():
    out_base = model_base_v6(x)
    out_ae16 = model_ae16(x)
    out_ae256 = model_ae256(x)
    p_tree = tree_member(x)

print("Base output shape  :", out_base.shape)
print("AE16 output shape  :", out_ae16.shape)
print("AE256 output shape :", out_ae256.shape)
print("CBTree output shape:",p_tree.shape)
print(p_tree.min(), p_tree.max())

Base output shape  : torch.Size([256])
AE16 output shape  : torch.Size([256])
AE256 output shape : torch.Size([256])
CBTree output shape: torch.Size([256])
tensor(1.2129e-06, device='cuda:0') tensor(0.9977, device='cuda:0')


In [21]:
# init the weighted ensemble with equal we will finetune by grid search (It can be abitary since we overwriting it anyways)
weighted_ensemble = WeightedFraudEnsemble(
    members=members,
    weights=[1/4, 1/4, 1/4, 1/4],
).to(DEVICE)

# init the stacked ensemble
stacker = StackingFraudMLP(
    input_dim=len(members),
    hidden_dims=[32, 16, 8],
    dropout=0.1,
    use_norm=True,
).to(DEVICE)

stacked_ensemble = StackedFraudEnsemble(
    members=members,
    stacker=stacker,
).to(DEVICE)

# We only want to optimise the stacker's params (The weight mixer) not the entire stacked ensembles params
# The members are meant to stay frozen
optimizer = torch.optim.AdamW(
    stacked_ensemble.stacker.parameters(),
    lr=1e-3,
    weight_decay=1e-3,
)

In [9]:
# "train" the weighted ensemble
from Utils.TrainUtils import threshold_sweep, compute_metrics_from_probs

val_labels = collect_labels(val_loader)
test_labels = collect_labels(test_loader)

weight_candidates = []
step = 0.1

values = [round(i * step, 2) for i in range(int(1 / step) + 1)]

for w_base in values:
    for w_ae16 in values:
        for w_ae256 in values:
            w_tree = 1.0 - w_base - w_ae16 - w_ae256
            if w_tree < 0:
                continue
            w_tree = round(w_tree, 2)
            if w_tree > 1:
                continue
            weight_candidates.append((w_base, w_ae16, w_ae256, w_tree))

print("Num candidate weight sets:", len(weight_candidates))

best_result = None
all_results = []

for i, weights in enumerate(weight_candidates):

    if i % 20 == 0:
        print(f"Testing {i}/{len(weight_candidates)}")

    ensemble = WeightedFraudEnsemble(
        members=members,
        weights=list(weights),
    ).to(DEVICE)

    val_probs = collect_weighted_ensemble_probs(
        ensemble=ensemble,
        loader=val_loader,
        device=DEVICE,
    )

    best_metrics, _ = threshold_sweep(
        probs=val_probs,
        labels=val_labels,
        num_thresholds=200,
        optimize_for="f2",
    )

    result = {
        "weights": weights,
        "best_threshold": best_metrics["threshold"],
        "precision": best_metrics["precision"],
        "recall": best_metrics["recall"],
        "f1": best_metrics["f1"],
        "f2": best_metrics["f2"],
        "accuracy": best_metrics["accuracy"],
        "tp": best_metrics["tp"],
        "fp": best_metrics["fp"],
        "tn": best_metrics["tn"],
        "fn": best_metrics["fn"],
    }
    all_results.append(result)

    if best_result is None or result["f2"] > best_result["f2"]:
        best_result = result

print("Best val result:")
print(best_result)

Num candidate weight sets: 269
Testing 0/269
Testing 20/269
Testing 40/269
Testing 60/269
Testing 80/269
Testing 100/269
Testing 120/269
Testing 140/269
Testing 160/269
Testing 180/269
Testing 200/269
Testing 220/269
Testing 240/269
Testing 260/269
Best val result:
{'weights': (0.2, 0.0, 0.0, 0.8), 'best_threshold': 0.5125376582145691, 'precision': 0.6839007986520224, 'recall': 0.7871311078820168, 'f1': 0.7318938321783257, 'f2': 0.7640649926616832, 'accuracy': 0.9798150844987673, 'tp': 1627, 'fp': 752, 'tn': 56235, 'fn': 440}


In [10]:
print("Best val result:")
for k, v in best_result.items():
    print(f"{k}: {v}")

Best val result:
weights: (0.2, 0.0, 0.0, 0.8)
best_threshold: 0.5125376582145691
precision: 0.6839007986520224
recall: 0.7871311078820168
f1: 0.7318938321783257
f2: 0.7640649926616832
accuracy: 0.9798150844987673
tp: 1627
fp: 752
tn: 56235
fn: 440


In [11]:
# eval the weighted ensem on test 

best_weights = list(best_result["weights"])
best_threshold = best_result["best_threshold"]

best_weighted_ensemble = WeightedFraudEnsemble(
    members=members,
    weights=best_weights,
).to(DEVICE)

test_probs = collect_weighted_ensemble_probs(
    ensemble=best_weighted_ensemble,
    loader=test_loader,
    device=DEVICE,
)

test_metrics = compute_metrics_from_probs(
    probs=test_probs,
    labels=test_labels,
    threshold=best_threshold,
)

print("=== Weighted Ensemble Test ===")
print("weights:", best_weights)
print("threshold:", best_threshold)
for k, v in test_metrics.items():
    print(f"{k}: {v}")

top_results = sorted(all_results, key=lambda x: x["f2"], reverse=True)[:10]

for r in top_results:
    print(r)

=== Weighted Ensemble Test ===
weights: [0.2, 0.0, 0.0, 0.8]
threshold: 0.5125376582145691
threshold: 0.5125376582145691
tp: 1555
fp: 804
tn: 56184
fn: 511
precision: 0.6591776176317966
recall: 0.7526621490767055
f1: 0.7028248537758075
f2: 0.7319024735972943
accuracy: 0.9777322450636744
{'weights': (0.2, 0.0, 0.0, 0.8), 'best_threshold': 0.5125376582145691, 'precision': 0.6839007986520224, 'recall': 0.7871311078820168, 'f1': 0.7318938321783257, 'f2': 0.7640649926616832, 'accuracy': 0.9798150844987673, 'tp': 1627, 'fp': 752, 'tn': 56235, 'fn': 440}
{'weights': (0.2, 0.0, 0.1, 0.7), 'best_threshold': 0.5125376582145691, 'precision': 0.686813186810284, 'recall': 0.7861635220087753, 'f1': 0.7331378249314625, 'f2': 0.7640586775406296, 'accuracy': 0.9799674873842619, 'tp': 1625, 'fp': 741, 'tn': 56246, 'fn': 442}
{'weights': (0.1, 0.2, 0.0, 0.7), 'best_threshold': 0.4924774169921875, 'precision': 0.6881355932174232, 'recall': 0.7856797290721544, 'f1': 0.7336796878128049, 'f2': 0.764019568781

In [ ]:
weighted_save = {
    "weights": best_weights,
    "threshold": best_threshold,
    "members": [
        "GatedMLP_V6",
        "GatedMLP_AE16_V4",
        "GatedMLP_AE256_V1",
        "CatBoost"
    ],
    "input_dim": input_dim,
}

torch.save(weighted_save, "checkpoints/WeightedEnsemble_V1_WITH_TREE.pt")

In [22]:
# training the stacker

import copy
import torch

criterion = torch.nn.BCELoss()

epochs = 50

best_state = None
best_loss = float("inf")

for epoch in range(epochs):
    stacked_ensemble.train()
    total_loss = 0.0

    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).float()

        optimizer.zero_grad()

        probs = stacked_ensemble(x_batch)   # already sigmoided probabilities
        loss = criterion(probs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / max(len(val_loader), 1)

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = copy.deepcopy(stacked_ensemble.stacker.state_dict())

    print(f"Epoch {epoch+1}/{epochs} - stacker_train_loss: {avg_loss:.6f}")

# restore best stacker
if best_state is not None:
    stacked_ensemble.stacker.load_state_dict(best_state)

stacked_ensemble.eval()
print("Best stacker train loss:", best_loss)

Epoch 1/50 - stacker_train_loss: 0.183956
Epoch 2/50 - stacker_train_loss: 0.078147
Epoch 3/50 - stacker_train_loss: 0.065612
Epoch 4/50 - stacker_train_loss: 0.062442
Epoch 5/50 - stacker_train_loss: 0.060401
Epoch 6/50 - stacker_train_loss: 0.058666
Epoch 7/50 - stacker_train_loss: 0.059251
Epoch 8/50 - stacker_train_loss: 0.057534
Epoch 9/50 - stacker_train_loss: 0.057353
Epoch 10/50 - stacker_train_loss: 0.056579
Epoch 11/50 - stacker_train_loss: 0.055823
Epoch 12/50 - stacker_train_loss: 0.055735
Epoch 13/50 - stacker_train_loss: 0.056262
Epoch 14/50 - stacker_train_loss: 0.055635
Epoch 15/50 - stacker_train_loss: 0.056116
Epoch 16/50 - stacker_train_loss: 0.055255
Epoch 17/50 - stacker_train_loss: 0.055048
Epoch 18/50 - stacker_train_loss: 0.055109
Epoch 19/50 - stacker_train_loss: 0.054922
Epoch 20/50 - stacker_train_loss: 0.055252
Epoch 21/50 - stacker_train_loss: 0.053934
Epoch 22/50 - stacker_train_loss: 0.054474
Epoch 23/50 - stacker_train_loss: 0.054459
Epoch 24/50 - stacke

In [23]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# collect labels + probs
test_labels = collect_labels(test_loader)

val_probs = collect_stacked_ensemble_probs(
    ensemble=stacked_ensemble,
    loader=val_loader,
    device=DEVICE,
)

val_labels = collect_labels(val_loader)

val_best_metrics, _ = threshold_sweep(
    probs=val_probs,
    labels=val_labels,
    num_thresholds=200,
    optimize_for="f2",
)

best_threshold = val_best_metrics["threshold"]

test_probs = collect_stacked_ensemble_probs(
    ensemble=stacked_ensemble,
    loader=test_loader,
    device=DEVICE,
)

test_labels = collect_labels(test_loader)

labels_np = test_labels.cpu().numpy()
probs_np = test_probs.cpu().numpy()

preds_np = (probs_np >= best_threshold).astype(int)

# sklearn metrics
acc  = accuracy_score(labels_np, preds_np)
prec = precision_score(labels_np, preds_np)
rec  = recall_score(labels_np, preds_np)
f1   = f1_score(labels_np, preds_np)

# compute F2 manually
beta = 2
f2 = (1 + beta**2) * (prec * rec) / ((beta**2 * prec) + rec + 1e-12)

print("=== FINAL RESULTS (OPTIMISED THRESHOLD) ===")
print(f"Threshold: {best_threshold:.6f}")
print(f"ACCURACY: {acc}")
print(f"PRECISION: {prec}")
print(f"RECALL: {rec}")
print(f"F1: {f1}")
print(f"F2: {f2}")
print()

print(classification_report(labels_np, preds_np, digits=2))

stacked_opti_metrics = compute_metrics_from_probs(
    probs=test_probs,
    labels=test_labels,
    threshold=best_threshold,
)

print(f"=== Stacked Ensemble Test ({best_threshold} threshold) ===")
for k, v in stacked_opti_metrics.items():
    print(f"{k}: {v}")

=== FINAL RESULTS (OPTIMISED THRESHOLD) ===
Threshold: 0.147281
ACCURACY: 0.9773766383310191
PRECISION: 0.6489795918367347
RECALL: 0.7696030977734754
F1: 0.704162976085031
F2: 0.7420197871941047

              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99     56988
         1.0       0.65      0.77      0.70      2066

    accuracy                           0.98     59054
   macro avg       0.82      0.88      0.85     59054
weighted avg       0.98      0.98      0.98     59054

=== Stacked Ensemble Test (0.14728137850761414 threshold) ===
threshold: 0.14728137850761414
tp: 1590
fp: 860
tn: 56128
fn: 476
precision: 0.6489795918340858
recall: 0.7696030977697502
f1: 0.7041629711180639
f2: 0.7420197849860932
accuracy: 0.9773766383308536


In [24]:
stacked_fixed_metrics = compute_metrics_from_probs(
    probs=test_probs,
    labels=test_labels,
    threshold=0.5,
)

print("=== Stacked Ensemble Test (0.5 threshold) ===")
for k, v in stacked_fixed_metrics.items():
    print(f"{k}: {v}")

=== Stacked Ensemble Test (0.5 threshold) ===
threshold: 0.5
tp: 1336
fp: 223
tn: 56765
fn: 730
precision: 0.8569595894749393
recall: 0.6466602129687965
f1: 0.7371034433696023
f2: 0.6800366470091973
accuracy: 0.9838622277913462


In [25]:
stacked_save = {
    "stacker_state_dict": stacked_ensemble.stacker.state_dict(),
    "stacker_config": stacked_ensemble.stacker.config,
    "member_names": [
        "GatedMLP_V6",
        "GatedMLP_AE16_V4",
        "GatedMLP_AE256_V1",
        "CatBoost"
    ],
    "notes": "4-model stacked ensemble tree integration (made net work 32-16-8)",
}

torch.save(stacked_save, "checkpoints/StackedEnsemble_FULL_TREE_V2.pt")

In [27]:
# inspecting the stacker due to unusually low threshold

stacked_test_probs = collect_stacked_ensemble_probs(
    ensemble=stacked_ensemble,
    loader=test_loader,
    device=DEVICE,
)

y_test = collect_labels(test_loader)

pos_probs = stacked_test_probs[y_test == 1]
neg_probs = stacked_test_probs[y_test == 0]

print("All probs:")
print(" min :", stacked_test_probs.min().item())
print(" max :", stacked_test_probs.max().item())
print(" mean:", stacked_test_probs.mean().item())

print("\nPositive probs:")
print(" mean:", pos_probs.mean().item())
print(" p25 :", pos_probs.quantile(0.25).item())
print(" p50 :", pos_probs.quantile(0.50).item())
print(" p75 :", pos_probs.quantile(0.75).item())

print("\nNegative probs:")
print(" mean:", neg_probs.mean().item())
print(" p25 :", neg_probs.quantile(0.25).item())
print(" p50 :", neg_probs.quantile(0.50).item())
print(" p75 :", neg_probs.quantile(0.75).item())

meta_x_test = stacked_ensemble.member_probs(next(iter(test_loader))[0].to(DEVICE))
print(meta_x_test[:5])

All probs:
 min : 0.0007435533334501088
 max : 0.9732306599617004
 mean: 0.035206545144319534

Positive probs:
 mean: 0.6525478363037109
 p25 : 0.19021916389465332
 p50 : 0.9625820517539978
 p75 : 0.9710204601287842

Negative probs:
 mean: 0.012825923040509224
 p25 : 0.0009107660735026002
 p50 : 0.0015217768959701061
 p75 : 0.00634286692366004
tensor([[2.3306e-04, 3.7277e-04, 6.6908e-04, 7.0656e-01],
        [1.2367e-03, 6.9306e-02, 5.6016e-02, 3.9634e-03],
        [9.9835e-01, 6.7035e-01, 7.1299e-01, 5.1299e-01],
        [1.6925e-04, 3.9800e-04, 9.7772e-04, 3.3885e-03],
        [9.9913e-03, 8.7435e-03, 3.3153e-02, 5.0544e-03]], device='cuda:0')


for the last inspection, we can see quite an interesting distribution which MAYBE explainable by:
```
[base, ae16, ae256, tree]

[0.0002, 0.0003, 0.0006, 0.706]  <- tree-only fraud
[0.0012, 0.069 , 0.056 , 0.003]  <- neural disagreement
[0.998 , 0.670 , 0.712 , 0.512]  <- consensus fraud
```